## 0. Pull latest code from GitHub

In [ ]:
import os, getpass
os.chdir("/content/Huang-Lab-Work")
token = getpass.getpass("Paste your GitHub token: ")
!git pull https://{token}@github.com/racyun/Huang-Lab-Work.git main
print("Done.")

# Cellpose-SAM Batch Segmentation

Runs Cellpose-SAM on ~6000 cell images stored in Google Drive and saves masks back to Drive.

**Running on CPU.** Expected time: ~30–120 sec/image — plan for multiple Colab sessions across several days.  
The skip-if-exists logic means each session picks up exactly where the last one left off.

**Before running:**
- Make sure Google Drive is mounted (cell 2)

**Resumable:** if Colab disconnects, just re-run from cell 5 onward — already-processed images are skipped automatically.

## 1. Check runtime

In [1]:
import torch

if torch.cuda.is_available():
    print(f"GPU detected: {torch.cuda.get_device_name(0)} — runtime will be fast.")
else:
    print("No GPU detected — running on CPU.")
    print("Expected ~30–120 sec per image. For 6000 images this will take many sessions.")
    print("Tip: skip-if-exists is enabled, so each new session resumes where the last left off.")


RuntimeError: No GPU detected. Go to Runtime → Change runtime type → T4 GPU and re-run.

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted.')

## 3. Install dependencies

In [ ]:
# Step 1: downgrade numpy to a version compatible with cellpose/scipy
!pip install -q "numpy<2.0" "cellpose" tifffile tqdm

# Step 2: restart the runtime so the new numpy is loaded
# After the restart, re-run from cell 4 (Config) onward — skip this cell.
import importlib.metadata
print(f"numpy version: {importlib.metadata.version("numpy")}")
print(f"cellpose version: {importlib.metadata.version("cellpose")}")
print("")
print("*** Now go to Runtime → Restart session, then re-run from cell 4 onward. ***")


## 4. Config

In [ ]:
from pathlib import Path

ROOT = Path('/content/drive/My Drive/Fusion AI/Prof Huang Project/Cellpose feature extractions')

# (input tiles folder, output masks folder) — one pair per stiffness condition
FOLDER_PAIRS = [
    (ROOT / 'imgs/260513_TC_Level/tiles', ROOT / 'masks/260513_TC_Level'),
    (ROOT / 'imgs/260514_900kPa/tiles',   ROOT / 'masks/260514_900kPa'),
    (ROOT / 'imgs/260516_5kPa/tiles',     ROOT / 'masks/260516_5kPa'),
    (ROOT / 'imgs/260516_500kPa/tiles',   ROOT / 'masks/260516_500kPa'),
    (ROOT / 'imgs/260521_150kPa/tiles',   ROOT / 'masks/260521_150kPa'),
    (ROOT / 'imgs/260522_500kPa/tiles',   ROOT / 'masks/260522_500kPa'),
]

MODEL_TYPE    = 'cpsam'  # Cellpose-SAM
DIAMETER      = None     # None = auto-estimate per image
CHANNELS      = [0, 0]   # [0,0] = use all channels for SAM
SKIP_EXISTING = True     # set False to reprocess everything from scratch
BATCH_SIZE    = 8        # images processed per model.eval() call
                         # increase if you have more RAM; decrease if you get OOM errors
IMG_EXTS      = {'.tif', '.tiff', '.png', '.jpg', '.jpeg'}

print('Config:')
for in_dir, out_dir in FOLDER_PAIRS:
    n = len([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS]) if in_dir.exists() else "?"
    print(f'  {in_dir.parent.name}  →  {n} images')


## 5. Load model (once — reused across all 6000 images)

In [ ]:
from cellpose import models

model = models.CellposeModel(gpu=False, model_type=MODEL_TYPE)
print(f"Model loaded: {MODEL_TYPE}  |  GPU={model.gpu}")


## 6. Segmentation functions

In [ ]:
import numpy as np
import tifffile
from tqdm.notebook import tqdm


def _load_image(img_path: Path) -> np.ndarray:
    """Load image and ensure shape is (H, W, 3)."""
    img = tifffile.imread(str(img_path))
    if img.ndim == 2:
        img = np.stack([img, img, img], axis=-1)
    elif img.ndim == 3 and img.shape[0] in (1, 3):  # (C,H,W) -> (H,W,C)
        img = np.moveaxis(img, 0, -1)
        if img.shape[-1] == 1:
            img = np.concatenate([img, img, img], axis=-1)
    return img


def segment_folder(in_dir: Path, out_dir: Path, model) -> dict:
    """Process all images in in_dir in batches, writing masks to out_dir."""
    out_dir.mkdir(parents=True, exist_ok=True)

    all_paths = sorted([p for p in in_dir.iterdir() if p.suffix.lower() in IMG_EXTS])
    if not all_paths:
        print(f'  [WARN] No images found in {in_dir}')
        return {'processed': 0, 'skipped': 0, 'failed': 0}

    # Split into todo (needs processing) and skipped
    todo, skipped_paths = [], []
    for p in all_paths:
        out_path = out_dir / f'{p.stem}_masks.tif'
        if SKIP_EXISTING and out_path.exists():
            skipped_paths.append(p)
        else:
            todo.append(p)

    processed = skipped = failed = 0
    skipped = len(skipped_paths)
    failed_files = []

    # Process in batches
    chunks = [todo[i:i+BATCH_SIZE] for i in range(0, len(todo), BATCH_SIZE)]
    pbar = tqdm(total=len(todo), desc=in_dir.parent.name, unit='img')

    for chunk in chunks:
        # Load all images in this chunk
        imgs, paths, load_errors = [], [], []
        for p in chunk:
            try:
                imgs.append(_load_image(p))
                paths.append(p)
            except Exception as e:
                print(f'  [FAIL load] {p.name}: {e}')
                failed_files.append(p.name)
                failed += 1
                pbar.update(1)

        if not imgs:
            continue

        # Run Cellpose on the whole batch at once
        try:
            masks_list, _, _ = model.eval(
                imgs,
                diameter=DIAMETER,
                channels=CHANNELS,
                normalize=True,
            )
        except Exception as e:
            print(f'  [FAIL batch] {[p.name for p in paths]}: {e}')
            failed += len(paths)
            failed_files += [p.name for p in paths]
            pbar.update(len(paths))
            continue

        # Save each mask
        for p, mask in zip(paths, masks_list):
            out_path = out_dir / f'{p.stem}_masks.tif'
            try:
                tifffile.imwrite(str(out_path), mask.astype(np.uint16), compression='lzw')
                processed += 1
            except Exception as e:
                print(f'  [FAIL save] {p.name}: {e}')
                failed += 1
                failed_files.append(p.name)
            pbar.update(1)

    pbar.close()
    if failed_files:
        print(f'  Failed files: {failed_files}')
    return {'processed': processed, 'skipped': skipped, 'failed': failed}


print('Functions defined.')


## 7. Run segmentation on all folders

Expected time on CPU: ~30–120 sec/image.  
If Colab disconnects, re-run cells 5 and 7 — already-processed images are skipped.


In [ ]:
import time

grand_total = {'processed': 0, 'skipped': 0, 'failed': 0}
t_start = time.time()

for in_dir, out_dir in FOLDER_PAIRS:
    print(f'\n── {in_dir.parent.name} ──')
    if not in_dir.exists():
        print(f'  [SKIP] Input folder not found: {in_dir}')
        continue

    t0 = time.time()
    stats = segment_folder(in_dir, out_dir, model)
    elapsed = time.time() - t0

    print(f'  processed={stats["processed"]}  skipped={stats["skipped"]}  '
          f'failed={stats["failed"]}  ({elapsed/60:.1f} min)')

    for k in grand_total:
        grand_total[k] += stats[k]

total_elapsed = time.time() - t_start
print(f'\n══ DONE ══')
print(f'Total processed : {grand_total["processed"]}')
print(f'Total skipped   : {grand_total["skipped"]}')
print(f'Total failed    : {grand_total["failed"]}')
print(f'Wall time       : {total_elapsed/60:.1f} min')

## 8. Sanity check — compare one mask to website output

Optional: visually verify that the programmatic masks match what you got from cellpose.org.
Set `SAMPLE_IMAGE` to any image path you already processed manually.

In [ ]:
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

# Change these two paths to a real image and its manually-downloaded mask
SAMPLE_IMAGE = ROOT / 'imgs/260513_TC_Level/tiles/YOUR_IMAGE.tif'
MANUAL_MASK  = ROOT / 'masks_manual/YOUR_IMAGE_masks.tif'  # your manually downloaded mask

if SAMPLE_IMAGE.exists():
    img  = tifffile.imread(str(SAMPLE_IMAGE))
    auto_mask_path = ROOT / f'masks/260513_TC_Level/{SAMPLE_IMAGE.stem}_masks.tif'
    auto_mask = tifffile.imread(str(auto_mask_path))

    ncols = 3 if MANUAL_MASK.exists() else 2
    fig, axes = plt.subplots(1, ncols, figsize=(5 * ncols, 5))

    axes[0].imshow(img if img.ndim == 3 else img, cmap='gray')
    axes[0].set_title('Original image')
    axes[0].axis('off')

    axes[1].imshow(auto_mask, cmap='tab20b')
    axes[1].set_title(f'Auto mask ({auto_mask.max()} cells)')
    axes[1].axis('off')

    if MANUAL_MASK.exists():
        manual_mask = tifffile.imread(str(MANUAL_MASK))
        axes[2].imshow(manual_mask, cmap='tab20b')
        axes[2].set_title(f'Manual mask ({manual_mask.max()} cells)')
        axes[2].axis('off')

    plt.tight_layout()
    plt.show()
else:
    print('Set SAMPLE_IMAGE and MANUAL_MASK paths above to run this cell.')